# v4 repaired: line following, gaps, and green-marker turns

Based on the team's v4 green-marker design and the earlier gap controller. No motor movement starts automatically; uncomment the driving call deliberately. Run one cell at a time on the Jetson, never Run All. Shut down previous camera kernels first. Resolve power alarms before driving.

Copy working calibration from your robot: black threshold, bands, speeds, gains, reverse setting, and gap timeout. The defaults here are trial settings, not your measured values. Green-triggered turns reverse one wheel regardless of the ordinary-curve ALLOW_REVERSE setting.

Corrections: valid green detector with area/shape filtering and return value; OpenCV 3/4 contour compatibility; actual stop/advance/turn sequence; turn timers start with the turn; departure and multi-frame reacquisition; latched timeout stops; marker cooldown; restored gap handling and camera freshness checks.

Limits: marker geometry is heuristic. There is no full intersection topology detector or confirmation that a marker is before an intersection. U-turn angle is not measured: minimum duration and line reacquisition need calibration and can select the wrong branch. White gaps use bounded open-loop motion. Software timeouts do not protect against a frozen process/driver. Keep a physical stop accessible. Hi Loubna

## 1. Initialize camera once


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 2. Black line and green detection
Green mask should show only physical tape markers. Drawn overlay dots are never fed into detection. Tune HSV, area bounds, and trigger position while stationary.


In [ ]:

# ============================================================
# 2. BLACK LINE AND GREEN DETECTION
# ============================================================

BLACK_THRESHOLD = 70

BANDS = [
    ("near", 0.72, 0.88),
    ("middle", 0.54, 0.70),
    ("far", 0.36, 0.52)
]

# Green HSV settings
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)

GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000

GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92

GREEN_TRIGGER_Y = 0.55
GREEN_MAX_DISTANCE_RATIO = 0.38
GREEN_SIDE_MARGIN = 8.0

# U-turn detection enabled
ENABLE_UTURN = True

GREEN_PAIR_HEIGHT_RATIO = 0.05
GREEN_PAIR_DISTANCE_RATIO = 0.90


# ============================================================
# BLACK LINE DETECTION
# ============================================================

def detect_curve(frame, green_mask=None):

    height, width = frame.shape[:2]

    gray = cv2.GaussianBlur(
        cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY),
        (5, 5),
        0
    )

    _, mask = cv2.threshold(
        gray,
        BLACK_THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )

    # Exclude detected green pixels from black line detection
    if green_mask is not None:
        mask[green_mask > 0] = 0

    # Overlay is only for display, never for detection
    overlay = frame.copy()

    positions = {}

    for name, top, bottom in BANDS:

        y0 = int(top * height)
        y1 = int(bottom * height)

        strip = mask[y0:y1, :]

        contours = cv2.findContours(
            strip,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )[-2]

        positions[name] = None

        if contours:

            contour = max(
                contours,
                key=cv2.contourArea
            )

            fraction = (
                cv2.contourArea(contour)
                / float(strip.size)
            )

            moments = cv2.moments(contour)

            if (
                0.005 < fraction < 0.80
                and moments["m00"] > 0
            ):

                cx = moments["m10"] / moments["m00"]

                cy = (
                    int(moments["m01"] / moments["m00"])
                    + y0
                )

                positions[name] = (
                    cx - width / 2.0
                ) / (width / 2.0)

                cv2.circle(
                    overlay,
                    (int(cx), cy),
                    4,
                    (0, 255, 0),
                    -1
                )

        cv2.rectangle(
            overlay,
            (0, y0),
            (width - 1, y1),
            (255, 0, 0),
            1
        )

        cv2.putText(
            overlay,
            name,
            (3, y0 + 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (0, 0, 255),
            1
        )

    cv2.line(
        overlay,
        (width // 2, 0),
        (width // 2, height - 1),
        (0, 255, 255),
        1
    )

    return positions, overlay, mask


# ============================================================
# GREEN MARKER DETECTION
# ============================================================

def detect_green(frame):

    height, width = frame.shape[:2]

    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    mask = cv2.inRange(
        hsv,
        GREEN_LOWER,
        GREEN_UPPER
    )

    # Limit detection to the floor region
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0

    kernel = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )[-2]

    markers = []

    for contour in contours:

        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)

        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA:
            continue

        if h == 0:
            continue

        # Reject extremely thin or elongated objects
        if not 0.4 <= w / float(h) <= 2.5:
            continue

        moments = cv2.moments(contour)

        if moments["m00"] <= 0:
            continue

        markers.append({
            "x": moments["m10"] / moments["m00"],
            "y": moments["m01"] / moments["m00"],
            "area": area,
            "w": w,
            "h": h,
            "box": (x, y, w, h)
        })

    markers.sort(
        key=lambda item: item["area"],
        reverse=True
    )

    return markers, mask


# ============================================================
# LINE POINTS
# ============================================================

def get_line_points(positions, width, height):

    band_y = {
        name: (top + bottom) / 2.0
        for name, top, bottom in BANDS
    }

    return [
        (
            width * (1.0 + positions[name]) / 2.0,
            height * band_y[name]
        )
        for name in ("far", "middle", "near")
        if positions.get(name) is not None
    ]


# ============================================================
# GREEN DIRECTION CLASSIFICATION
# LEFT / RIGHT / BOTH / NONE / UNKNOWN
# ============================================================

def classify_green_direction(
    positions,
    markers,
    width,
    height,
    black_mask=None
):

    if not markers:
        return "NONE"

    if positions.get("near") is None:
        return "UNKNOWN"

    points = get_line_points(
        positions,
        width,
        height
    )

    if len(points) < 2:
        return "UNKNOWN"

    near_x, near_y = points[-1]
    far_x, far_y = points[0]

    travel_dx = far_x - near_x
    travel_dy = far_y - near_y

    length = np.hypot(
        travel_dx,
        travel_dy
    )

    if length < 1e-6:
        return "UNKNOWN"

    travel_dx /= length
    travel_dy /= length

    max_distance = width * GREEN_MAX_DISTANCE_RATIO

    left_candidates = []
    right_candidates = []

    for marker in markers:

        # Ignore distant markers
        if marker["y"] < height * GREEN_TRIGGER_Y:
            continue

        dx = marker["x"] - near_x
        dy = marker["y"] - near_y

        distance = np.hypot(dx, dy)

        if distance > max_distance:
            continue

        # Signed lateral position relative to travel direction
        side_score = (
            dx * travel_dy
            - dy * travel_dx
        )

        if abs(side_score) <= GREEN_SIDE_MARGIN:
            continue

        # Image coordinates:
        # x increases right, y increases downward
        if side_score > 0:
            right_candidates.append((distance, marker))
        else:
            left_candidates.append((distance, marker))

    if not left_candidates and not right_candidates:
        return "NONE"

    left_candidates.sort(
        key=lambda item: item[0]
    )

    right_candidates.sort(
        key=lambda item: item[0]
    )

    # U-turn: green markers on both sides
    if (
        ENABLE_UTURN
        and left_candidates
        and right_candidates
    ):

        left_distance, left_marker = left_candidates[0]
        right_distance, right_marker = right_candidates[0]

        y_difference = abs(
            left_marker["y"] - right_marker["y"]
        )

        combined_distance = (
            left_distance + right_distance
        )

        same_junction_height = (
            y_difference <= height * GREEN_PAIR_HEIGHT_RATIO
        )

        close_pair = (
            combined_distance
            <= max_distance * GREEN_PAIR_DISTANCE_RATIO
        )

        if same_junction_height and close_pair:
            return "BOTH"

    if left_candidates and not right_candidates:
        return "LEFT"

    if right_candidates and not left_candidates:
        return "RIGHT"

    if left_candidates[0][0] < right_candidates[0][0]:
        return "LEFT"

    return "RIGHT"


## 3. Existing curve controller
Use measured reliable motor commands. Default curve speed is lower than base speed; raise it if a wheel stalls.


In [ ]:

# ============================================================
# 3. CURVE CONTROLLER
# ============================================================

BASE_SPEED = 0.15
CURVE_SPEED = 0.13
KP = 0.14
MAX_SPEED = 0.25

TURN_SPEED = 0.15
ALLOW_REVERSE = False

TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0

LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5


def clamp(value, low, high):
    return max(low, min(high, value))


class CurveController:

    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0

        if direction > 0:
            return speed, inner
        return inner, speed

    def step(self, positions, now):

        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"

        near = positions.get("near")

        visible = [
            (name, value)
            for name, value in positions.items()
            if name in ("near", "middle", "far")
            and value is not None
        ]

        # ----------------------------------------------------
        # LINE LOST
        # ----------------------------------------------------

        if not visible:

            if self.lost_started is None:
                self.lost_started = now

            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")

            if self.turn_started is None:

                if self.last_direction == 0:
                    return self.stop(
                        "line lost without known direction"
                    )

                self.turn_started = now
                self.turn_direction = self.last_direction

            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None

        # ----------------------------------------------------
        # CALCULATE LINE ERROR
        # ----------------------------------------------------

        weights = {
            "near": 0.45,
            "middle": 0.35,
            "far": 0.20
        }

        error = (
            sum(weights[name] * value for name, value in visible)
            / sum(weights[name] for name, _ in visible)
        )

        farthest = positions.get("far")

        if farthest is None:
            farthest = positions.get("middle")

        # Predict upcoming curve
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)

        error = clamp(error, -1.0, 1.0)

        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        # ----------------------------------------------------
        # EXIT TURN
        # ----------------------------------------------------

        if self.turn_started is not None:

            if (
                near is not None
                and abs(near) < TURN_EXIT
                and abs(error) < TURN_EXIT
            ):
                self.turn_started = None
                self.turn_direction = 0

            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        # ----------------------------------------------------
        # ENTER TURN
        # ----------------------------------------------------

        if (
            self.turn_started is None
            and abs(error) >= TURN_ENTER
        ):
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        # ----------------------------------------------------
        # NORMAL LINE FOLLOWING
        # ----------------------------------------------------

        bend = min(abs(error) / TURN_ENTER, 1.0)

        base = (
            BASE_SPEED
            + (CURVE_SPEED - BASE_SPEED) * bend
        )

        correction = KP * error

        left = clamp(
            base + correction,
            0.0,
            MAX_SPEED
        )

        right = clamp(
            base - correction,
            0.0,
            MAX_SPEED
        )

        return left, right, "FOLLOW"


## 4. Gap controller
Copy your calibrated gap timeout and speed here.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now)

        left, right, state = super().step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


## 5. Green turn state machine
Three matching processed frames request STOP → ADVANCE → TURN → FOLLOW. The small advance moves toward the junction; calibrate GREEN_TRIGGER_Y and GREEN_ADVANCE_TIME together. Set advance time to zero if already correctly positioned. Minimum turn duration begins after advance and is not a measured angle. A timeout stops, and the same marker is ignored until absent for several frames after cooldown.


In [ ]:

# ============================================================
# 5. GREEN NAVIGATOR
# LEFT / RIGHT / U-TURN
# ============================================================

# Green marker confirmation
GREEN_CONFIRM_FRAMES = 3

# Stop briefly before turning
GREEN_STOP_TIME = 0.12

# Move into junction
GREEN_ADVANCE_TIME = 0.15
GREEN_ADVANCE_SPEED = BASE_SPEED

# Normal green turns
GREEN_TURN_SPEED = 0.15
GREEN_TURN_MIN_TIME = 0.30
GREEN_TURN_TIMEOUT = 1.50

# Line reacquisition
GREEN_REACQUIRE_FRAMES = 3
GREEN_HEADING_EXIT = 0.18

# Prevent repeated triggering
GREEN_CLEAR_FRAMES = 5
GREEN_COOLDOWN = 0.8

# U-turn enabled
ENABLE_UTURN = True

UTURN_MIN_TIME = 0.8
UTURN_TIMEOUT = 2.5

# These detection settings belong in section 2:
# GREEN_TRIGGER_Y
# GREEN_MAX_DISTANCE_RATIO
# GREEN_SIDE_MARGIN


def line_centered(positions):

    near = positions.get("near")
    middle = positions.get("middle")

    return (
        near is not None
        and middle is not None
        and abs(near) < TURN_EXIT
        and abs(middle) < TURN_EXIT
        and abs(near - middle) < GREEN_HEADING_EXIT
    )


class GreenNavigator:

    def __init__(self):

        self.line = GapCurveController()

        self.state = "FOLLOW"
        self.state_started = None

        self.direction = None

        self.candidate = None
        self.candidate_frames = 0

        self.armed = True
        self.clear_frames = 0
        self.cooldown_until = 0.0

        self.departed = False
        self.reacquire_frames = 0

        self.stopped = False
        self.reason = ""

    def stop(self, reason):

        self.stopped = True
        self.reason = reason

        return 0.0, 0.0, "STOP: " + reason

    def enter(self, state, now):

        self.state = state
        self.state_started = now

    # --------------------------------------------------------
    # TURN COMMAND
    # --------------------------------------------------------

    def turn(self):

        speed = clamp(
            GREEN_TURN_SPEED,
            0.0,
            MAX_SPEED
        )

        if self.direction == "RIGHT":
            return speed, -speed, "GREEN_TURN_RIGHT"

        if self.direction == "LEFT":
            return -speed, speed, "GREEN_TURN_LEFT"

        if self.direction == "BOTH" and ENABLE_UTURN:
            return -speed, speed, "GREEN_U_TURN"

        return 0.0, 0.0, "STOP: invalid turn direction"

    # --------------------------------------------------------
    # MAIN NAVIGATION
    # --------------------------------------------------------

    def step(self, positions, direction, now):

        if self.stopped:
            return 0.0, 0.0, "STOP: " + self.reason

        # ----------------------------------------------------
        # GREEN STOP
        # ----------------------------------------------------

        if self.state == "GREEN_STOP":

            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, "GREEN_STOP"

            self.enter("GREEN_ADVANCE", now)

        # ----------------------------------------------------
        # GREEN ADVANCE
        # ----------------------------------------------------

        if self.state == "GREEN_ADVANCE":

            if now - self.state_started < GREEN_ADVANCE_TIME:

                speed = clamp(
                    GREEN_ADVANCE_SPEED,
                    0.0,
                    MAX_SPEED
                )

                return speed, speed, "GREEN_ADVANCE"

            self.departed = False
            self.reacquire_frames = 0

            self.enter("GREEN_TURN", now)

        # ----------------------------------------------------
        # GREEN TURN
        # ----------------------------------------------------

        if self.state == "GREEN_TURN":

            elapsed = now - self.state_started

            is_uturn = (
                self.direction == "BOTH"
                and ENABLE_UTURN
            )

            if is_uturn:
                minimum = UTURN_MIN_TIME
                timeout = UTURN_TIMEOUT
            else:
                minimum = GREEN_TURN_MIN_TIME
                timeout = GREEN_TURN_TIMEOUT

            if elapsed >= timeout:
                return self.stop("green turn timed out")

            centered = line_centered(positions)

            # Robot must leave the original line
            if not centered:
                self.departed = True

            # Reacquire a centered line after minimum time
            if (
                self.departed
                and elapsed >= minimum
                and centered
            ):
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0

            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:

                left, right, state = self.turn()

                if state.startswith("STOP:"):
                    return self.stop("invalid turn direction")

                return left, right, state

            # Reset line and gap history
            self.line = GapCurveController()

            self.enter("FOLLOW", now)

            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0

            self.departed = False
            self.reacquire_frames = 0

            return 0.0, 0.0, "GREEN_DONE"

        # ----------------------------------------------------
        # GREEN MARKER RE-ARM
        # ----------------------------------------------------

        if not self.armed:

            if direction == "NONE":
                self.clear_frames += 1
            else:
                self.clear_frames = 0

            if (
                now >= self.cooldown_until
                and self.clear_frames >= GREEN_CLEAR_FRAMES
            ):
                self.armed = True

        # ----------------------------------------------------
        # GREEN MARKER CONFIRMATION
        # ----------------------------------------------------

        eligible = (
            self.armed
            and self.line.gap_started is None
            and self.line.turn_started is None
            and direction in ("LEFT", "RIGHT", "BOTH")
        )

        if direction == "BOTH" and not ENABLE_UTURN:
            eligible = False

        if eligible:

            if direction == self.candidate:
                self.candidate_frames += 1
            else:
                self.candidate = direction
                self.candidate_frames = 1

            if self.candidate_frames >= GREEN_CONFIRM_FRAMES:

                self.direction = direction

                self.armed = False
                self.clear_frames = 0

                self.candidate = None
                self.candidate_frames = 0

                self.enter("GREEN_STOP", now)

                return 0.0, 0.0, "GREEN_STOP"

        else:

            self.candidate = None
            self.candidate_frames = 0

        # ----------------------------------------------------
        # NORMAL LINE FOLLOWING
        # ----------------------------------------------------

        left, right, state = self.line.step(
            positions,
            now
        )

        if self.line.stopped:
            return self.stop(state)

        return left, right, state

## 6. Runner
Preview computes commands but only stops motors. Every run creates fresh controller state. Camera frames are checked for freshness even when no new frame arrives.


In [ ]:

# ============================================================
# 6. MAIN LOOP
# CAMERA PREVIEW / ROBOT DRIVE
# ============================================================

overlay_view = widgets.Image(
    format="jpeg",
    width=336,
    height=336
)

mask_view = widgets.Image(
    format="jpeg",
    width=336,
    height=336
)

status_view = widgets.Label(
    value="Ready - motors stopped"
)

display(
    widgets.VBox([
        widgets.HBox([
            overlay_view,
            mask_view
        ]),
        status_view
    ])
)


# ============================================================
# MAIN ROBOT LOOP
# ============================================================

def run_v5(seconds=10, drive=False):

    if seconds <= 0:
        raise ValueError("seconds must be positive")

    robot.stop()

    # New controller for each run
    controller = GreenNavigator()

    deadline = time.monotonic() + seconds

    previous_stamp = None
    next_display = 0.0

    try:

        while time.monotonic() < deadline:

            # ------------------------------------------------
            # CAMERA FRAME
            # ------------------------------------------------

            sample = latest["sample"]

            if sample is None:
                print("STOP: no camera frame")
                break

            frame, captured_at = sample
            now = time.monotonic()

            # Stop if camera data is too old
            if now - captured_at > FRAME_TIMEOUT:
                print("STOP: stale camera frame")
                break

            # Wait for a new camera frame
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue

            previous_stamp = captured_at

            # ------------------------------------------------
            # GREEN DETECTION
            # ------------------------------------------------

            markers, green_mask = detect_green(frame)

            # ------------------------------------------------
            # BLACK LINE DETECTION
            # ------------------------------------------------

            positions, overlay, black_mask = detect_curve(
                frame,
                green_mask
            )

            height, width = frame.shape[:2]

            # ------------------------------------------------
            # GREEN DIRECTION
            # ------------------------------------------------

            direction = classify_green_direction(
                positions,
                markers,
                width,
                height,
                black_mask
            )

            # LEFT / RIGHT / BOTH / NONE / UNKNOWN

            # ------------------------------------------------
            # FRAME SAFETY CHECK
            # ------------------------------------------------

            now = time.monotonic()

            if now >= deadline:
                print("STOP: time limit reached")
                break

            if now - captured_at > FRAME_TIMEOUT:
                print("STOP: slow image processing")
                break

            # ------------------------------------------------
            # ROBOT CONTROLLER
            # ------------------------------------------------

            left, right, state = controller.step(
                positions,
                direction,
                now
            )

            # Stop immediately if controller fails
            if controller.stopped or state.startswith("STOP:"):
                robot.stop()
                print(state)
                break

            # ------------------------------------------------
            # MOTOR CONTROL
            # ------------------------------------------------

            if drive:
                robot.set_motors(left, right)
            else:
                robot.stop()

            # ------------------------------------------------
            # DISPLAY
            # ------------------------------------------------

            if now >= next_display:

                # Draw detected black line
                points = get_line_points(
                    positions,
                    width,
                    height
                )

                for a, b in zip(points, points[1:]):

                    cv2.line(
                        overlay,
                        tuple(map(int, a)),
                        tuple(map(int, b)),
                        (255, 0, 255),
                        2
                    )

                # Draw green marker boxes
                for marker in markers:

                    x, y, w, h = marker["box"]

                    cv2.rectangle(
                        overlay,
                        (x, y),
                        (x + w, y + h),
                        (0, 255, 0),
                        2
                    )

                # Display direction and state
                cv2.putText(
                    overlay,
                    "DIR: " + direction,
                    (5, 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (0, 255, 255),
                    1
                )

                cv2.putText(
                    overlay,
                    "STATE: " + state,
                    (5, 38),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.40,
                    (255, 255, 0),
                    1
                )

                # Overlay image
                overlay_view.value = bgr8_to_jpeg(overlay)

                # Green mask image
                mask_view.value = bgr8_to_jpeg(
                    cv2.cvtColor(
                        green_mask,
                        cv2.COLOR_GRAY2BGR
                    )
                )

                # Status text
                status_view.value = (
                    "%s | %s | marker=%s | L=%.2f R=%.2f"
                    % (
                        "DRIVE" if drive else "PREVIEW",
                        state,
                        direction,
                        left,
                        right
                    )
                )

                next_display = now + 0.10

            time.sleep(0.01)

    except KeyboardInterrupt:
        print("STOP: interrupted")

    finally:

        robot.stop()

        status_view.value = "STOPPED - motors off"

        print("Run ended; motors stopped.")


# ============================================================
# RUN COMMANDS
# ============================================================

def run_preview(seconds=20):
    run_v5(
        seconds=seconds,
        drive=False
    )


def run_robot(seconds=2):
    run_v5(
        seconds=seconds,
        drive=True
    )


def emergency_stop():
    robot.stop()
    print("EMERGENCY STOP")


print("Main loop loaded.")
print("Use run_preview(20) to test detection.")
print("Use run_robot(2) for short driving test.")

## 7. Stationary preview
Test no marker, left, right, and both. Move/rotate the robot by hand to simulate the manoeuvre. A timeout stops preview too; reposition and run again.


In [ ]:
run_preview(seconds=20)


## 8. Short driving test
Begin with a single left marker, then right, then both. Start far enough before the marker for detection. Initial two-second tests may end before a whole turn completes; extend the overall duration gradually, not the failure timeout. Retest straight lines, curves, and gaps.


In [ ]:
# Explicitly uncomment to move the robot:
run_robot(seconds=30)


## 9. Cleanup
Run before switching notebooks. Shut down this kernel afterward.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()
